# MobileNetV5 con KerasHub para PlantVillage

Experimento para comparar MobileNetV5 contra el MobileNetV2 actual de la app Android. La comparación mantiene las mismas 38 clases de PlantVillage, registra precisión, tamaño de exportación TFLite y tiempo promedio de inferencia local.

Fuente de arquitectura: [KerasHub MobileNetV5](https://keras.io/keras_hub/api/models/mobilenetv5/).

In [ ]:
from pathlib import Path
import json
import os
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
import tensorflow_datasets as tfds
import keras
import keras_hub
from keras_hub.models import MobileNetV5Backbone, MobileNetV5ImageClassifier
from sklearn.metrics import classification_report, confusion_matrix

SEED = 123
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

PROJECT_DIR = Path('/Users/carolinachavez/convolutional-neuronal-network/MobileNetV2')
ANDROID_ASSETS_DIR = Path('/Users/carolinachavez/Documents/Leaf-disease-applicaton/Leafdisease/app/src/main/assets')
LOG_DOC = Path('/Users/carolinachavez/Documents/Tesis docs/context/mobilenetv5_keras.md')

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE
NUM_CLASSES_EXPECTED = 38

MODEL_NAME = 'mobilenetv5_keras_v1'
PRESET = 'mobilenetv5_300m_enc_gemma3n'
USE_PRETRAINED_WEIGHTS = True

# Training safety controls.
# The pretrained MobileNetV5 preset has ~294M frozen backbone params. A full epoch
# over 43k PlantVillage images can be very slow on CPU, even when only the head trains.
TRAINING_MODE = 'smoke_test'  # 'smoke_test' first, then 'full' for the real run.
SMOKE_TRAIN_EXAMPLES = 1024
SMOKE_VAL_EXAMPLES = 256
SMOKE_TEST_EXAMPLES = 256

HEAD_EPOCHS = 2 if TRAINING_MODE == 'smoke_test' else 8
FINE_TUNE_EPOCHS = 0 if TRAINING_MODE == 'smoke_test' else 8
FINE_TUNE_LR = 1e-5
HEAD_LR = 1e-3

RUN_TRAINING = True
RUN_FINE_TUNING = TRAINING_MODE == 'full'
RUN_EXPORT = TRAINING_MODE == 'full'

print('Python executable:', os.sys.executable)
print('TensorFlow:', tf.__version__)
print('Keras:', keras.__version__)
print('KerasHub:', keras_hub.__version__)
print('Available MobileNetV5 presets:', list(MobileNetV5Backbone.presets.keys()))


## Dataset y preprocessing

Se usa `tensorflow_datasets` con `plant_village`, el mismo universo de 38 clases que espera la app. El preprocessing normaliza imágenes a `[-1, 1]`, alineado con el clasificador Android actual (`PIXEL_MEAN = 127.5`, `PIXEL_STD = 127.5`).

In [ ]:
def preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    image = (image - 127.5) / 127.5
    return image, label

raw_ds, info = tfds.load(
    'plant_village',
    split='train',
    as_supervised=True,
    with_info=True,
)

class_names = info.features['label'].names
num_classes = info.features['label'].num_classes
total_examples = info.splits['train'].num_examples

assert num_classes == NUM_CLASSES_EXPECTED, f'Expected 38 classes, got {num_classes}'

train_size = int(total_examples * 0.80)
val_size = int(total_examples * 0.10)
test_size = total_examples - train_size - val_size

shuffled = raw_ds.shuffle(
    buffer_size=min(total_examples, 10_000),
    seed=SEED,
    reshuffle_each_iteration=False,
)

train_raw = shuffled.take(train_size)
remaining = shuffled.skip(train_size)
val_raw = remaining.take(val_size)
test_raw = remaining.skip(val_size)

if TRAINING_MODE == 'smoke_test':
    train_raw = train_raw.take(SMOKE_TRAIN_EXAMPLES)
    val_raw = val_raw.take(SMOKE_VAL_EXAMPLES)
    test_raw = test_raw.take(SMOKE_TEST_EXAMPLES)
    effective_train_size = SMOKE_TRAIN_EXAMPLES
    effective_val_size = SMOKE_VAL_EXAMPLES
    effective_test_size = SMOKE_TEST_EXAMPLES
else:
    effective_train_size = train_size
    effective_val_size = val_size
    effective_test_size = test_size

train_ds = train_raw.map(preprocess, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
val_ds = val_raw.map(preprocess, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_ds = test_raw.map(preprocess, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_steps = int(np.ceil(effective_train_size / BATCH_SIZE))
val_steps = int(np.ceil(effective_val_size / BATCH_SIZE))
test_steps = int(np.ceil(effective_test_size / BATCH_SIZE))

print('Total examples:', total_examples)
print('Full split sizes:', {'train': train_size, 'validation': val_size, 'test': test_size})
print('Training mode:', TRAINING_MODE)
print('Effective split sizes:', {'train': effective_train_size, 'validation': effective_val_size, 'test': effective_test_size})
print('Estimated steps:', {'train': train_steps, 'validation': val_steps, 'test': test_steps})
print('Number of classes:', num_classes)
print('First labels:', class_names[:5])


In [3]:
labels_path = PROJECT_DIR / 'labels.txt'
labels_path.write_text('\n'.join(class_names) + '\n')

android_labels_path = ANDROID_ASSETS_DIR / 'labels.txt'
if android_labels_path.exists():
    android_labels = [line.strip() for line in android_labels_path.read_text().splitlines() if line.strip()]
    labels_match_android = android_labels == class_names
else:
    android_labels = []
    labels_match_android = False

print('Wrote labels:', labels_path)
print('Labels match Android asset:', labels_match_android)
if not labels_match_android:
    print('WARNING: Android labels differ from TFDS labels or labels file is missing.')


Wrote labels: /Users/carolinachavez/convolutional-neuronal-network/MobileNetV2/labels.txt
Labels match Android asset: True


## Baseline actual MobileNetV2

La app Android usa `mobilenetv2_v3_fp16.tflite`. Este bloque registra su tamaño y permite compararlo con las exportaciones de MobileNetV5.

In [4]:
def file_size_mb(path):
    path = Path(path)
    return path.stat().st_size / (1024 * 1024) if path.exists() else None

baseline_tflite = PROJECT_DIR / 'mobilenetv2_v3_fp16.tflite'
android_baseline_tflite = ANDROID_ASSETS_DIR / 'mobilenetv2_v3_fp16.tflite'

baseline_info = {
    'model': 'mobilenetv2_v3_fp16',
    'project_tflite': str(baseline_tflite),
    'project_size_mb': file_size_mb(baseline_tflite),
    'android_tflite': str(android_baseline_tflite),
    'android_size_mb': file_size_mb(android_baseline_tflite),
    'num_classes': len(class_names),
}

pd.DataFrame([baseline_info])

,model,project_tflite,project_size_mb,android_tflite,android_size_mb,num_classes
0,mobilenetv2_v3_fp16,/Users/carolinachavez/convolutional-neuronal-n...,4.349388,/Users/carolinachavez/Documents/Leaf-disease-a...,4.349388,38


## Construcción del modelo MobileNetV5

Se intenta cargar el preset oficial con pesos preentrenados. Si la descarga o el preset fallan, el notebook cae a una inicialización aleatoria del mismo preset; si eso también falla, usa una configuración compacta compatible con la API de `MobileNetV5Backbone` para dejar el experimento ejecutable y registrar la decisión.

In [5]:
def build_custom_lightweight_backbone():
    model_args = {
        'stackwise_block_types': [['er'], ['uir', 'uir'], ['uir', 'uir']],
        'stackwise_num_blocks': [1, 2, 2],
        'stackwise_num_filters': [[24], [48, 48], [96, 96]],
        'stackwise_strides': [[2], [2, 1], [2, 1]],
        'stackwise_act_layers': [['relu'], ['relu', 'relu'], ['gelu', 'gelu']],
        'stackwise_exp_ratios': [[4.0], [6.0, 6.0], [6.0, 6.0]],
        'stackwise_se_ratios': [[0.0], [0.0, 0.0], [0.25, 0.25]],
        'stackwise_dw_kernel_sizes': [[0], [5, 5], [5, 5]],
        'stackwise_dw_start_kernel_sizes': [[0], [0, 0], [0, 0]],
        'stackwise_dw_end_kernel_sizes': [[0], [0, 0], [0, 0]],
        'stackwise_exp_kernel_sizes': [[3], [0, 0], [0, 0]],
        'stackwise_pw_kernel_sizes': [[1], [0, 0], [0, 0]],
        'stackwise_num_heads': [[0], [0, 0], [0, 0]],
        'stackwise_key_dims': [[0], [0, 0], [0, 0]],
        'stackwise_value_dims': [[0], [0, 0], [0, 0]],
        'stackwise_kv_strides': [[0], [0, 0], [0, 0]],
        'stackwise_use_cpe': [[False], [False, False], [False, False]],
        'use_msfa': False,
        'image_shape': (*IMG_SIZE, 3),
    }
    return MobileNetV5Backbone(**model_args)


def build_mobilenetv5_classifier(num_classes):
    attempts = []

    if USE_PRETRAINED_WEIGHTS:
        try:
            backbone = MobileNetV5Backbone.from_preset(
                PRESET,
                load_weights=True,
                image_shape=(*IMG_SIZE, 3),
            )
            source = f'{PRESET}: pretrained weights'
            attempts.append('Loaded preset with pretrained weights.')
        except Exception as exc:
            attempts.append(f'Pretrained preset failed: {type(exc).__name__}: {exc}')
            backbone = None
    else:
        backbone = None
        attempts.append('Pretrained weights disabled by USE_PRETRAINED_WEIGHTS=False.')

    if backbone is None:
        try:
            backbone = MobileNetV5Backbone.from_preset(
                PRESET,
                load_weights=False,
                image_shape=(*IMG_SIZE, 3),
            )
            source = f'{PRESET}: random weights'
            attempts.append('Loaded preset architecture with random weights.')
        except Exception as exc:
            attempts.append(f'Random preset failed: {type(exc).__name__}: {exc}')
            backbone = build_custom_lightweight_backbone()
            source = 'custom lightweight MobileNetV5-style config: random weights'
            attempts.append('Loaded custom lightweight fallback.')

    model = MobileNetV5ImageClassifier(
        backbone=backbone,
        num_classes=num_classes,
        preprocessor=None,
        global_pool='avg',
        drop_rate=0.2,
    )
    return model, source, attempts

model, model_source, build_attempts = build_mobilenetv5_classifier(num_classes)
model.backbone.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=HEAD_LR),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

print('Model source:', model_source)
print('\n'.join(build_attempts))
model.summary()


100%|██████████| 30.0k/30.0k [00:00<00:00, 708kB/s]


100%|██████████| 1.10G/1.10G [00:40<00:00, 29.1MB/s]


Model source: mobilenetv5_300m_enc_gemma3n: pretrained weights
Loaded preset with pretrained weights.


Model: "mobile_net_v5_image_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                  ┃ Output Shape                       ┃             Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)                      │ (None, 224, 224, 3)                │                   0 │
├───────────────────────────────────────────────┼────────────────────────────────────┼─────────────────────┤
│ mobile_net_v5_backbone (MobileNetV5Backbone)  │ (None, 16, 16, 2048)               │         294,284,096 │
├───────────────────────────────────────────────┼────────────────────────────────────┼─────────────────────┤
│ select_adaptive_pool2d (SelectAdaptivePool2d) │ (None, 2048)                       │                   0 │
├───────────────────────────────────────────────┼────────────────────────────────────┼─────────────────────┤
│ dropout_66 (Dropout)                          │ (None, 2048)                       │                   0 │
├───────────────────────────────────────────────┼────────────────────────────────────┼─────────────────────┤
│ classifier (Dense)                            │ (None, 38)                         │              77,862 │
└───────────────────────────────────────────────┴────────────────────────────────────┴─────────────────────┘

 Total params: 294,361,958 (1.10 GB)

 Trainable params: 77,862 (304.15 KB)

 Non-trainable params: 294,284,096 (1.10 GB)

## Entrenamiento

Primero se entrena la cabeza de clasificación con el backbone congelado; luego se hace fine-tuning con learning rate bajo. Los checkpoints quedan en formato `.keras`.

In [ ]:
checkpoint_path = PROJECT_DIR / 'mobilenetv5_keras_best.keras'
log_dir = PROJECT_DIR / 'logs' / MODEL_NAME

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(checkpoint_path),
        monitor='val_accuracy',
        mode='max',
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        mode='max',
        patience=3,
        restore_best_weights=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=2,
        min_lr=1e-7,
        verbose=1,
    ),
    keras.callbacks.CSVLogger(str(PROJECT_DIR / f'{MODEL_NAME}_training_log.csv')),
]

history_head = None
history_finetune = None

if RUN_TRAINING:
    print(f'Training mode: {TRAINING_MODE}. Head epochs: {HEAD_EPOCHS}.')
    print(f'Expected train steps per epoch: {train_steps}; validation steps: {val_steps}.')
    history_head = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=HEAD_EPOCHS,
        callbacks=callbacks,
    )
else:
    print('RUN_TRAINING=False: skipping head training.')


In [ ]:
if RUN_TRAINING and RUN_FINE_TUNING:
    model.backbone.trainable = True

    for layer in model.backbone.layers:
        if isinstance(layer, keras.layers.BatchNormalization):
            layer.trainable = False

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=FINE_TUNE_LR),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )

    history_finetune = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=HEAD_EPOCHS + FINE_TUNE_EPOCHS,
        initial_epoch=history_head.epoch[-1] + 1 if history_head else 0,
        callbacks=callbacks,
    )
elif RUN_TRAINING:
    print('RUN_FINE_TUNING=False: skipping fine-tuning for smoke test.')
else:
    print('RUN_TRAINING=False: skipping fine-tuning.')


## Evaluación

Este bloque carga el mejor checkpoint si existe, evalúa el test set y produce `classification_report` y matriz de confusión.

In [ ]:
if checkpoint_path.exists():
    eval_model = keras.models.load_model(checkpoint_path)
    print('Loaded best checkpoint:', checkpoint_path)
else:
    eval_model = model
    print('Checkpoint not found; using in-memory model.')

test_loss, test_accuracy = eval_model.evaluate(test_ds, verbose=1)
print({'test_loss': test_loss, 'test_accuracy': test_accuracy})

y_true = []
y_pred = []
y_conf = []

for images, labels in test_ds:
    probs = eval_model.predict(images, verbose=0)
    preds = np.argmax(probs, axis=1)
    y_true.extend(labels.numpy().tolist())
    y_pred.extend(preds.tolist())
    y_conf.extend(np.max(probs, axis=1).tolist())

report_dict = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report_dict).transpose()
report_df.head()

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(16, 14))
sns.heatmap(cm, cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('MobileNetV5 - Matriz de confusión PlantVillage')
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
errors = []
for true_idx, pred_idx, conf in zip(y_true, y_pred, y_conf):
    if true_idx != pred_idx:
        errors.append({
            'true_label': class_names[true_idx],
            'predicted_label': class_names[pred_idx],
            'confidence': conf,
        })

errors_df = pd.DataFrame(errors).sort_values('confidence', ascending=False)
errors_df.head(25)

## Exportación a SavedModel y TFLite

Se exporta el mejor modelo a SavedModel y se generan variantes TFLite FP32 y FP16. Si TFLite falla por ops no soportadas, el error queda visible para decidir si conviene insistir con esta arquitectura en móvil.

In [ ]:
saved_model_dir = PROJECT_DIR / 'saved_models' / MODEL_NAME
fp32_tflite_path = PROJECT_DIR / f'{MODEL_NAME}_fp32.tflite'
fp16_tflite_path = PROJECT_DIR / f'{MODEL_NAME}_fp16.tflite'

export_results = {}

if RUN_EXPORT:
    saved_model_dir.parent.mkdir(parents=True, exist_ok=True)
    eval_model.export(str(saved_model_dir))
    export_results['saved_model'] = str(saved_model_dir)

    try:
        converter = tf.lite.TFLiteConverter.from_saved_model(str(saved_model_dir))
        tflite_fp32 = converter.convert()
        fp32_tflite_path.write_bytes(tflite_fp32)
        export_results['fp32_tflite'] = str(fp32_tflite_path)
        export_results['fp32_size_mb'] = file_size_mb(fp32_tflite_path)
    except Exception as exc:
        export_results['fp32_error'] = f'{type(exc).__name__}: {exc}'

    try:
        converter = tf.lite.TFLiteConverter.from_saved_model(str(saved_model_dir))
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]
        tflite_fp16 = converter.convert()
        fp16_tflite_path.write_bytes(tflite_fp16)
        export_results['fp16_tflite'] = str(fp16_tflite_path)
        export_results['fp16_size_mb'] = file_size_mb(fp16_tflite_path)
    except Exception as exc:
        export_results['fp16_error'] = f'{type(exc).__name__}: {exc}'
else:
    export_results['export'] = 'Skipped because RUN_EXPORT=False'

export_results

## Medición simple de inferencia TFLite

La métrica es local, no Android real. Sirve como filtro inicial junto con tamaño del archivo; la decisión móvil final debería validarse en la app o en un dispositivo objetivo.

In [ ]:
def benchmark_tflite(tflite_path, dataset, warmup=3, runs=25):
    tflite_path = Path(tflite_path)
    if not tflite_path.exists():
        return {'path': str(tflite_path), 'error': 'missing file'}

    interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    sample_images, _ = next(iter(dataset.unbatch().batch(1)))
    sample = sample_images.numpy().astype(input_details['dtype'])

    timings_ms = []
    for i in range(warmup + runs):
        interpreter.set_tensor(input_details['index'], sample)
        start = time.perf_counter()
        interpreter.invoke()
        _ = interpreter.get_tensor(output_details['index'])
        elapsed_ms = (time.perf_counter() - start) * 1000
        if i >= warmup:
            timings_ms.append(elapsed_ms)

    return {
        'path': str(tflite_path),
        'size_mb': file_size_mb(tflite_path),
        'avg_ms': float(np.mean(timings_ms)),
        'p95_ms': float(np.percentile(timings_ms, 95)),
        'runs': runs,
    }

benchmarks = []
for candidate in [baseline_tflite, fp32_tflite_path, fp16_tflite_path]:
    benchmarks.append(benchmark_tflite(candidate, test_ds))

pd.DataFrame(benchmarks)

## Comparación y registro de resultados

Este bloque prepara una tabla de comparación y agrega una entrada resumida a la bitácora Markdown. Después de cada corrida conviene revisar manualmente la conclusión, sobre todo si el modelo exportado supera el tamaño/latencia razonable para móvil.

In [ ]:
comparison_rows = [
    {
        'model': 'MobileNetV2 actual FP16',
        'test_accuracy': None,
        'tflite_size_mb': file_size_mb(baseline_tflite),
        'avg_inference_ms_local': next((b.get('avg_ms') for b in benchmarks if b.get('path') == str(baseline_tflite)), None),
        'notes': 'Modelo actual de la app; accuracy no medido en esta corrida.',
    },
    {
        'model': 'MobileNetV5 KerasHub',
        'test_accuracy': test_accuracy,
        'tflite_size_mb': file_size_mb(fp16_tflite_path),
        'avg_inference_ms_local': next((b.get('avg_ms') for b in benchmarks if b.get('path') == str(fp16_tflite_path)), None),
        'notes': model_source,
    },
]

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

In [ ]:
def append_run_to_log():
    timestamp = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
    fp16_size = file_size_mb(fp16_tflite_path)
    fp32_size = file_size_mb(fp32_tflite_path)
    avg_ms = next((b.get('avg_ms') for b in benchmarks if b.get('path') == str(fp16_tflite_path)), None)

    entry = f'''

## Corrida - {timestamp}

- Notebook: `{PROJECT_DIR / 'MobileNetKerasV5.ipynb'}`
- Ambiente: `learning_ai`
- TensorFlow: `{tf.__version__}`
- Keras: `{keras.__version__}`
- KerasHub: `{keras_hub.__version__}`
- Dataset: `tensorflow_datasets/plant_village`, 38 clases, split 80/10/10, seed `{SEED}`
- Modelo: `{model_source}`
- Test accuracy: `{test_accuracy:.4f}`
- Test loss: `{test_loss:.4f}`
- TFLite FP32: `{fp32_tflite_path.name}`, tamaño `{fp32_size}` MB
- TFLite FP16: `{fp16_tflite_path.name}`, tamaño `{fp16_size}` MB
- Inferencia local FP16 promedio: `{avg_ms}` ms
- Decisión provisional: comparar precisión contra tamaño/latencia antes de reemplazar MobileNetV2 en Android.
'''
    with LOG_DOC.open('a') as f:
        f.write(entry)

append_run_to_log()
print('Updated log:', LOG_DOC)